# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [1]:
import numpy as np
import pandas as pd

# Load data
transitions = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
rewards = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')

# Define states and actions (order matters for array indexing)
states = ['exposed-1', 'exposed-2', 'exposed-3', 'symptoms-1', 'symptoms-2', 'symptoms-3', 'recovered']
actions_list = ['do-nothing', 'drink-oj', 'sleep-8']

n_states = len(states)
n_actions = len(actions_list)

state_idx = {s: i for i, s in enumerate(states)}
action_idx = {a: i for i, a in enumerate(actions_list)}

# Build R array (n_actions x n_states)
R = np.zeros((n_actions, n_states))
for _, row in rewards.iterrows():
    a = action_idx[row['action']]
    s = state_idx[row['state']]
    R[a, s] = row['reward']

# Build P array (n_actions x n_states x n_states)
P = np.zeros((n_actions, n_states, n_states))
for _, row in transitions.iterrows():
    a = action_idx[row['action']]
    s = state_idx[row['state']]
    ns = state_idx[row['next_state']]
    P[a, s, ns] = row['probability']

# Helper functions (from examples)
def compute_qT_once(R, P, gamma, v):
    return R + gamma * P @ v

def iterate_values_once(R, P, gamma, v):
    return np.max(compute_qT_once(R, P, gamma, v), axis=0)

def value_iteration(R, P, gamma, max_iterations=10000, tolerance=1e-9):
    v_old = np.zeros(R.shape[-1])
    for i in range(max_iterations):
        v_new = iterate_values_once(R, P, gamma, v_old)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new
        v_old = v_new
    return v_old

def iterative_policy_evaluation(R, P, gamma, pi, max_iterations=10000, tolerance=1e-9):
    n = R.shape[-1]
    R_pi = R[pi, np.arange(n)]
    P_pi = P[pi, np.arange(n), :]
    R_pi = R_pi.reshape(1, *R_pi.shape)
    P_pi = P_pi.reshape(1, *P_pi.shape)
    v_old = np.zeros(R.shape[-1])
    for i in range(max_iterations):
        v_new = iterate_values_once(R_pi, P_pi, gamma, v_old)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new
        v_old = v_new
    return v_old

# Part 1: Evaluate the do-nothing policy using value iteration (gamma=1, no discounting)
gamma = 1.0
do_nothing_idx = action_idx['do-nothing']
pi_do_nothing = np.full(n_states, do_nothing_idx, dtype='int64')

v_do_nothing = iterative_policy_evaluation(R, P, gamma, pi_do_nothing)

# Expected discomfort = -expected_reward (rewards are negative for discomfort)
do_nothing_discomfort = -v_do_nothing
print("Do-nothing expected discomfort by state:")
for i, s in enumerate(states):
    print(f"  {s}: {do_nothing_discomfort[i]:.6f}")


Do-nothing expected discomfort by state:
  exposed-1: 3.413333
  exposed-2: 4.266667
  exposed-3: 5.333333
  symptoms-1: 6.666667
  symptoms-2: 5.000000
  symptoms-3: 1.666667
  recovered: -0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [2]:
do_nothing_df = pd.DataFrame({
    'state': states,
    'expected_discomfort': do_nothing_discomfort
})
do_nothing_df.to_csv('do-nothing-discomfort.tsv', sep='\t', index=False)
print(do_nothing_df.to_string(index=False))


     state  expected_discomfort
 exposed-1             3.413333
 exposed-2             4.266667
 exposed-3             5.333333
symptoms-1             6.666667
symptoms-2             5.000000
symptoms-3             1.666667
 recovered            -0.000000


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [3]:
# Part 2: Compute optimal treatment plan (maximize rewards = minimize discomfort)
v_optimal = value_iteration(R, P, gamma)

# Extract optimal policy as argmax over actions for each state
qT = compute_qT_once(R, P, gamma, v_optimal)
pi_optimal = np.argmax(qT, axis=0)

print("Optimal policy by state:")
for i, s in enumerate(states):
    print(f"  {s}: {actions_list[pi_optimal[i]]}")


Optimal policy by state:
  exposed-1: sleep-8
  exposed-2: sleep-8
  exposed-3: sleep-8
  symptoms-1: drink-oj
  symptoms-2: drink-oj
  symptoms-3: drink-oj
  recovered: do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [4]:
optimal_actions_df = pd.DataFrame({
    'state': states,
    'action': [actions_list[a] for a in pi_optimal]
})
optimal_actions_df.to_csv('minimum-discomfort-actions.tsv', sep='\t', index=False)
print(optimal_actions_df.to_string(index=False))


     state     action
 exposed-1    sleep-8
 exposed-2    sleep-8
 exposed-3    sleep-8
symptoms-1   drink-oj
symptoms-2   drink-oj
symptoms-3   drink-oj
 recovered do-nothing


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [5]:
# Part 3: Compute expected discomfort under the optimal policy
v_min_discomfort = iterative_policy_evaluation(R, P, gamma, pi_optimal)

# Expected discomfort = -expected_reward
min_discomfort = -v_min_discomfort
print("Expected discomfort under optimal policy by state:")
for i, s in enumerate(states):
    print(f"  {s}: {min_discomfort[i]:.6f}")


Expected discomfort under optimal policy by state:
  exposed-1: 0.750000
  exposed-2: 1.500000
  exposed-3: 3.000000
  symptoms-1: 6.000000
  symptoms-2: 4.500000
  symptoms-3: 1.500000
  recovered: -0.000000


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [7]:
min_discomfort_df = pd.DataFrame({
    'state': states,
    'expected_discomfort': min_discomfort
})
min_discomfort_df.to_csv('minimum-discomfort-values.tsv', sep='\t', index=False)
print(min_discomfort_df.to_string(index=False))


     state  expected_discomfort
 exposed-1                 0.75
 exposed-2                 1.50
 exposed-3                 3.00
symptoms-1                 6.00
symptoms-2                 4.50
symptoms-3                 1.50
 recovered                -0.00


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [8]:
# Part 4: Modify reward function to minimize days sick
# Sick states: symptoms-1, symptoms-2, symptoms-3 (exposed does NOT count)
# Reward = -1 if in a sick state, 0 otherwise; action does not matter
sick_states = {'symptoms-1', 'symptoms-2', 'symptoms-3'}

R_duration = np.zeros((n_actions, n_states))
for a in range(n_actions):
    for i, s in enumerate(states):
        if s in sick_states:
            R_duration[a, i] = -1.0

print("Duration reward function (non-zero entries):")
for a, action in enumerate(actions_list):
    for i, s in enumerate(states):
        if R_duration[a, i] != 0:
            print(f"  {action}, {s}: {R_duration[a, i]}")


Duration reward function (non-zero entries):
  do-nothing, symptoms-1: -1.0
  do-nothing, symptoms-2: -1.0
  do-nothing, symptoms-3: -1.0
  drink-oj, symptoms-1: -1.0
  drink-oj, symptoms-2: -1.0
  drink-oj, symptoms-3: -1.0
  sleep-8, symptoms-1: -1.0
  sleep-8, symptoms-2: -1.0
  sleep-8, symptoms-3: -1.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [9]:
# Save in same format as twizzleflu-rewards.tsv (action, state, reward)
duration_rewards_records = []
for a, action in enumerate(actions_list):
    for i, s in enumerate(states):
        duration_rewards_records.append({
            'action': action,
            'state': s,
            'reward': R_duration[a, i]
        })

duration_rewards_df = pd.DataFrame(duration_rewards_records)
duration_rewards_df.to_csv('duration-rewards.tsv', sep='\t', index=False)
print(duration_rewards_df.to_string(index=False))


    action      state  reward
do-nothing  exposed-1     0.0
do-nothing  exposed-2     0.0
do-nothing  exposed-3     0.0
do-nothing symptoms-1    -1.0
do-nothing symptoms-2    -1.0
do-nothing symptoms-3    -1.0
do-nothing  recovered     0.0
  drink-oj  exposed-1     0.0
  drink-oj  exposed-2     0.0
  drink-oj  exposed-3     0.0
  drink-oj symptoms-1    -1.0
  drink-oj symptoms-2    -1.0
  drink-oj symptoms-3    -1.0
  drink-oj  recovered     0.0
   sleep-8  exposed-1     0.0
   sleep-8  exposed-2     0.0
   sleep-8  exposed-3     0.0
   sleep-8 symptoms-1    -1.0
   sleep-8 symptoms-2    -1.0
   sleep-8 symptoms-3    -1.0
   sleep-8  recovered     0.0


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [10]:
# Part 5: Compute optimal policy to minimize Twizzleflu duration
v_duration_optimal = value_iteration(R_duration, P, gamma)

qT_duration = compute_qT_once(R_duration, P, gamma, v_duration_optimal)
pi_duration = np.argmax(qT_duration, axis=0)

print("Duration-minimizing policy by state:")
for i, s in enumerate(states):
    print(f"  {s}: {actions_list[pi_duration[i]]}")


Duration-minimizing policy by state:
  exposed-1: sleep-8
  exposed-2: sleep-8
  exposed-3: sleep-8
  symptoms-1: do-nothing
  symptoms-2: do-nothing
  symptoms-3: do-nothing
  recovered: do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [11]:
min_duration_actions_df = pd.DataFrame({
    'state': states,
    'action': [actions_list[a] for a in pi_duration]
})
min_duration_actions_df.to_csv('minimum-duration-actions.tsv', sep='\t', index=False)
print(min_duration_actions_df.to_string(index=False))


     state     action
 exposed-1    sleep-8
 exposed-2    sleep-8
 exposed-3    sleep-8
symptoms-1 do-nothing
symptoms-2 do-nothing
symptoms-3 do-nothing
 recovered do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [12]:
# Part 6: Expected number of days sick under the duration-minimizing policy
v_duration = iterative_policy_evaluation(R_duration, P, gamma, pi_duration)

# Expected sick days = -v_duration (reward is -1 per sick day, so total = -(sick days))
expected_sick_days = -v_duration
print("Expected sick days under duration-minimizing policy by state:")
for i, s in enumerate(states):
    print(f"  {s}: {expected_sick_days[i]:.6f}")


Expected sick days under duration-minimizing policy by state:
  exposed-1: 1.250000
  exposed-2: 2.500000
  exposed-3: 5.000000
  symptoms-1: 10.000000
  symptoms-2: 6.666667
  symptoms-3: 3.333333
  recovered: -0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [13]:
min_duration_days_df = pd.DataFrame({
    'state': states,
    'expected_sick_days': expected_sick_days
})
min_duration_days_df.to_csv('minimum-duration-days.tsv', sep='\t', index=False)
print(min_duration_days_df.to_string(index=False))


     state  expected_sick_days
 exposed-1            1.250000
 exposed-2            2.500000
 exposed-3            5.000000
symptoms-1           10.000000
symptoms-2            6.666667
symptoms-3            3.333333
 recovered           -0.000000


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [14]:
# Part 7: Compare expected discomfort under both policies
# speed_discomfort: expected discomfort when following pi_duration (minimize days sick)
# minimize_discomfort: expected discomfort when following pi_optimal (from Part 3)

# Evaluate pi_duration using the ORIGINAL discomfort reward function R
v_speed = iterative_policy_evaluation(R, P, gamma, pi_duration)
speed_discomfort = -v_speed

print(f"{'state':<15} {'speed_discomfort':>18} {'minimize_discomfort':>20}")
for i, s in enumerate(states):
    print(f"  {s:<15} {speed_discomfort[i]:>18.6f} {min_discomfort[i]:>20.6f}")


state             speed_discomfort  minimize_discomfort
  exposed-1                 0.833333             0.750000
  exposed-2                 1.666667             1.500000
  exposed-3                 3.333333             3.000000
  symptoms-1                6.666667             6.000000
  symptoms-2                5.000000             4.500000
  symptoms-3                1.666667             1.500000
  recovered                -0.000000            -0.000000


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [15]:
policy_comparison_df = pd.DataFrame({
    'state': states,
    'speed_discomfort': speed_discomfort,
    'minimize_discomfort': min_discomfort
})
policy_comparison_df.to_csv('policy-comparison.tsv', sep='\t', index=False)
print(policy_comparison_df.to_string(index=False))


     state  speed_discomfort  minimize_discomfort
 exposed-1          0.833333                 0.75
 exposed-2          1.666667                 1.50
 exposed-3          3.333333                 3.00
symptoms-1          6.666667                 6.00
symptoms-2          5.000000                 4.50
symptoms-3          1.666667                 1.50
 recovered         -0.000000                -0.00


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.